In [26]:
import requests
import pandas as pd
from tqdm import tqdm
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [27]:
# ================= CONFIG =================
INPUT_FILES = [
    "north_of_vietnam.csv",
    "central_of_vietnam.csv",
    "south_of_vietnam.csv"
]

START_DATE = "2025-06-01"
END_DATE = "2025-09-30"

OUTPUT_FILE = "aqi_air_weather_6_9_2025.csv"
MAX_THREADS = 10

In [28]:
# ================= LOAD =================
dfs = [pd.read_csv(f) for f in INPUT_FILES]
df = pd.concat(dfs, ignore_index=True)

In [29]:
# ================= API =================

def get_air_quality(lat, lon):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": [
            "pm2_5",
            "pm10",
            "carbon_monoxide",
            "nitrogen_dioxide",
            "us_aqi"
        ],
        "timezone": "Asia/Bangkok"
    }

    try:
        res = requests.get(url, params=params, timeout=30)
        if res.status_code == 200:
            return res.json().get("hourly", None)
    except:
        pass
    return None


def get_weather(lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": [
            "temperature_2m_mean",
            "relative_humidity_2m_mean",
            "windspeed_10m_max"
        ],
        "timezone": "Asia/Bangkok"
    }

    try:
        res = requests.get(url, params=params, timeout=30)
        if res.status_code == 200:
            return res.json().get("daily", None)
    except:
        pass
    return None

In [30]:
# ================= PROCESS =================
def process_location(row):
    lat = row["lat"]
    lon = row["lon"]
    district = row.get("district", "")
    province = row.get("city", "")

    try:
        # ===== AQI HOURLY =====
        air = get_air_quality(lat, lon)
        if air is None:
            return []

        air_df = pd.DataFrame({
            "datetime": air["time"],
            "pm2_5": air["pm2_5"],
            "pm10": air["pm10"],
            "co": air["carbon_monoxide"],
            "no2": air["nitrogen_dioxide"],
            "aqi": air["us_aqi"]
        })

        air_df["datetime"] = pd.to_datetime(air_df["datetime"])
        air_df["date"] = air_df["datetime"].dt.date

        # ===== WEATHER DAILY =====
        weather = get_weather(lat, lon)
        if weather is None:
            return []

        weather_df = pd.DataFrame({
            "date": weather["time"],
            "temp_mean": weather["temperature_2m_mean"],
            "humidity": weather["relative_humidity_2m_mean"],
            "wind_max": weather["windspeed_10m_max"]
        })

        weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.date
        # ===== MERGE theo DATE =====
        merged = pd.merge(air_df, weather_df, on="date", how="left")

        merged["district"] = district
        merged["province"] = province
        merged["lat"] = lat
        merged["lon"] = lon

        return merged.to_dict("records")

    except Exception as e:
        print(f"Lỗi {district}-{province}:", e)
        return []

In [31]:
# ================= MULTI THREAD =================
results = []

with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    futures = [executor.submit(process_location, row) for _, row in df.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures)):
        results.extend(future.result())

100%|██████████| 686/686 [00:52<00:00, 13.11it/s]


In [32]:
# ================= SAVE =================
output_df = pd.DataFrame(results)

output_df = output_df.sort_values(by=["province", "district", "datetime"])

output_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("✅ DONE:", OUTPUT_FILE)

✅ DONE: aqi_air_weather_6_9_2025.csv


In [34]:
df = pd.read_csv("/content/aqi_air_weather_6_9_2025.csv")

In [35]:
df.head()

,datetime,pm2_5,pm10,co,no2,aqi,date,temp_mean,humidity,wind_max,district,province,lat,lon
0,2025-06-01 00:00:00,37.6,39.6,304.0,34.2,106,2025-06-01,31.5,72,17.7,Bac Giang City,Bac Giang,21.2731,106.1946
1,2025-06-01 01:00:00,40.5,42.7,296.0,36.3,103,2025-06-01,31.5,72,17.7,Bac Giang City,Bac Giang,21.2731,106.1946
2,2025-06-01 02:00:00,43.3,45.3,290.0,38.8,104,2025-06-01,31.5,72,17.7,Bac Giang City,Bac Giang,21.2731,106.1946
3,2025-06-01 03:00:00,44.7,46.6,286.0,41.2,104,2025-06-01,31.5,72,17.7,Bac Giang City,Bac Giang,21.2731,106.1946
4,2025-06-01 04:00:00,43.1,44.5,290.0,41.5,105,2025-06-01,31.5,72,17.7,Bac Giang City,Bac Giang,21.2731,106.1946


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 799344 entries, 0 to 799343
Data columns (total 14 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   datetime   799344 non-null  object 
 1   pm2_5      799344 non-null  float64
 2   pm10       799344 non-null  float64
 3   co         799344 non-null  float64
 4   no2        799344 non-null  float64
 5   aqi        799344 non-null  int64  
 6   date       799344 non-null  object 
 7   temp_mean  799344 non-null  float64
 8   humidity   799344 non-null  int64  
 9   wind_max   799344 non-null  float64
 10  district   799344 non-null  object 
 11  province   799344 non-null  object 
 12  lat        799344 non-null  float64
 13  lon        799344 non-null  float64
dtypes: float64(8), int64(2), object(4)
memory usage: 85.4+ MB
